# Défi 2 NLP (medRxiv 2026) — Projet 1 : Discipline Prediction
**Approche :** Machine Learning Classique (TF-IDF hybride & Classifieurs linéaires)  
**Auteur :** Redouane Kenniche

---

### Objectif du notebook
Ce notebook implémente la chaîne complète de classification multi-classes (16 disciplines médicales — `category_grp`) à partir des résumés bruts (*abstracts*) de preprints medRxiv, avec optimisation de la métrique officielle **F1 macro**.

## 1. Importation des bibliothèques
Chargement des modules nécessaires pour le traitement de texte, la vectorisation TF-IDF, l'entraînement des classifieurs linéaires et l'évaluation des performances.

In [1]:
import html
import os
import re
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, f1_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import FeatureUnion
from sklearn.svm import LinearSVC

## 2. Fonctions de chargement et de nettoyage
Définition des utilitaires permettant de localiser les fichiers (compressés ou non), de gérer automatiquement le séparateur CSV (`;` ou `,`) et de nettoyer le texte brut des abstracts tout en préservant les expressions mathématiques (ex. `p<0.05`).

In [2]:
def trouver_fichier(noms_possibles):
    for nom in noms_possibles:
        if os.path.exists(nom):
            return nom
    return None


def nettoyer_colonnes(df):
    df.columns = (
        df.columns.astype(str)
        .str.replace("\ufeff", "", regex=False)
        .str.strip()
        .str.strip('"')
        .str.strip("'")
    )
    return df


def lire_csv_robuste(filepath, is_train=True):
    cibles_possibles = {"category_grp", "category_group", "discipline", "label"}
    for skip_bad in [False, True]:
        for engine in ["c", "python"]:
            if skip_bad and engine == "c":
                continue
            for enc in ["utf-8", "latin-1", "cp1252"]:
                for sep in [";", ",", "\t"]:
                    try:
                        kwargs = {"encoding": enc, "sep": sep, "engine": engine}
                        if skip_bad:
                            kwargs["on_bad_lines"] = "skip"
                        df = pd.read_csv(filepath, **kwargs)
                        df = nettoyer_colonnes(df)
                        cols_lower = {c.lower() for c in df.columns}
                        if "abstract" in cols_lower and len(df.columns) >= 2:
                            if not is_train or (cols_lower & cibles_possibles):
                                return df
                    except Exception:
                        continue
    raise ValueError(f"Impossible de parser : {filepath}")


def nettoyer_texte(texte):
    if not isinstance(texte, str):
        return ""
    texte = html.unescape(texte).lower()
    # Suppression ciblée des balises HTML uniquement
    texte = re.sub(
        r"</?(?:p|b|i|u|sub|sup|br|em|strong|span|div|ul|ol|li)\b[^>]*>",
        " ",
        texte,
    )
    texte = re.sub(r"[^a-z0-9\s\-]", " ", texte)
    return re.sub(r"\s+", " ", texte).strip()

## 3. Préparation des données et dédoublonnage
Chargement du jeu d'entraînement, sélection de la variable cible à 16 modalités (`category_grp`) et suppression des doublons issus des différentes versions d'un même preprint afin d'éviter toute fuite de données entre l'entraînement et la validation.

In [3]:
TRAIN_FILE = trouver_fichier([
    "ClassificationsMedicales_AbstractsTrainingSet.csv",
    "ClassificationsMedicales_AbstractsTrainingSet.csv.gz",
])
TEST_FILE = trouver_fichier([
    "ClassificationsMedicales_AbstractsTestSet.csv",
    "ClassificationsMedicales_AbstractsTestSet.csv.gz",
])

df_train = lire_csv_robuste(TRAIN_FILE, is_train=True)

col_map = {c.lower(): c for c in df_train.columns}
col_abstract = col_map["abstract"]
col_target = None
for candidate in ["category_grp", "category_group", "discipline", "label"]:
    if candidate in col_map:
        col_target = col_map[candidate]
        break

df_train = df_train.dropna(subset=[col_abstract, col_target]).reset_index(
    drop=True
)
df_train["clean_abstract"] = df_train[col_abstract].apply(nettoyer_texte)
df_train[col_target] = df_train[col_target].astype(str).str.strip()

# Suppression des versions successives d'un même abstract
avant_dedup = len(df_train)
df_train = df_train.drop_duplicates(subset=["clean_abstract"]).reset_index(
    drop=True
)
print(
    f"Cible : '{col_target}' ({df_train[col_target].nunique()} classes) |"
    f" Lignes uniques : {len(df_train)}/{avant_dedup}"
)

Cible : 'category_grp' (16 classes) | Lignes uniques : 9324/10958


## 4. Découpage stratifié et vectorisation TF-IDF hybride
Séparation des données en 80 % entraînement et 20 % validation en conservant les proportions de chaque discipline (`stratify=y`), suivie d'une représentation combinant les n-grammes de mots (1 à 2) et de caractères (3 à 5).

In [4]:
X = df_train["clean_abstract"].values
y = df_train[col_target].values

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Combinaison de TF-IDF sur les mots et sur les sous-mots (caractères)
features = FeatureUnion([
    (
        "word",
        TfidfVectorizer(
            ngram_range=(1, 2),
            min_df=2,
            max_df=0.90,
            sublinear_tf=True,
            stop_words="english",
        ),
    ),
    (
        "char",
        TfidfVectorizer(
            analyzer="char_wb",
            ngram_range=(3, 5),
            min_df=5,
            max_df=0.90,
            sublinear_tf=True,
        ),
    ),
])

X_train_vec = features.fit_transform(X_train)
X_val_vec = features.transform(X_val)

## 5. Entraînement et sélection du modèle
Comparaison de plusieurs classifieurs linéaires (`LinearSVC` et `LogisticRegression`) avec pondération automatique des classes (`class_weight="balanced"`) pour maximiser la métrique officielle **F1 macro**.

In [5]:
modeles = {
    "LinearSVC (C=0.3)": LinearSVC(
        C=0.3, class_weight="balanced", random_state=42
    ),
    "LinearSVC (C=0.6)": LinearSVC(
        C=0.6, class_weight="balanced", random_state=42
    ),
    "LinearSVC (C=1.0)": LinearSVC(
        C=1.0, class_weight="balanced", random_state=42
    ),
    "LogReg (C=3.0)": LogisticRegression(
        C=3.0, class_weight="balanced", max_iter=1000, random_state=42
    ),
}

best_name, best_model, best_f1 = None, None, -1.0
for name, clf in modeles.items():
    clf.fit(X_train_vec, y_train)
    preds = clf.predict(X_val_vec)
    f1 = f1_score(y_val, preds, average="macro")
    acc = accuracy_score(y_val, preds)
    print(f"{name:20s} | F1 macro : {f1:.4f} | Accuracy : {acc:.4f}")
    if f1 > best_f1:
        best_f1, best_name, best_model = f1, name, clf

print(f"\nMeilleur modèle ML : {best_name} (F1 macro = {best_f1:.4f})")
print(classification_report(y_val, best_model.predict(X_val_vec), digits=3))

LinearSVC (C=0.3)    | F1 macro : 0.5840 | Accuracy : 0.5952
LinearSVC (C=0.6)    | F1 macro : 0.5798 | Accuracy : 0.5909
LinearSVC (C=1.0)    | F1 macro : 0.5720 | Accuracy : 0.5850
LogReg (C=3.0)       | F1 macro : 0.5708 | Accuracy : 0.5850

Meilleur modèle ML : LinearSVC (C=0.3) (F1 macro = 0.5840)
                                              precision    recall  f1-score   support

          Appareil locomoteur & réadaptation      0.647     0.629     0.638        70
                Autres spécialités médicales      0.627     0.520     0.568       100
                  Cardio-métabolique & rénal      0.705     0.717     0.711       180
         Chirurgie, anesthésie & soins aigus      0.557     0.557     0.557        61
                       Génétique & génomique      0.626     0.680     0.652       128
                       Imagerie & diagnostic      0.646     0.609     0.627        69
                 Infectiologie & immunologie      0.565     0.651     0.605       146
       

## 6. Ré-entraînement complet et génération des prédictions sur le jeu de test
Le meilleur modèle retenu est ré-entraîné sur l'ensemble des abstracts d'entraînement avant de prédire les disciplines du jeu de test et d'exporter le fichier de soumission au format `(ID, pred)`.

In [6]:
if TEST_FILE:
    df_test = lire_csv_robuste(TEST_FILE, is_train=False)
    test_col_map = {c.lower(): c for c in df_test.columns}
    col_id_test = test_col_map.get("id", df_test.columns[0])
    col_abs_test = test_col_map["abstract"]

    # Ré-entraînement sur 100 % des données disponibles
    X_full_vec = features.fit_transform(X)
    best_model.fit(X_full_vec, y)

    X_test_vec = features.transform(
        df_test[col_abs_test].fillna("").apply(nettoyer_texte)
    )
    preds_test = best_model.predict(X_test_vec)

    df_sub = pd.DataFrame({"ID": df_test[col_id_test], "pred": preds_test})
    df_sub.to_csv("Projet1_Script1_ML.csv", index=False)
    print(
        f"Fichier généré : Projet1_Script1_ML.csv ({df_sub.shape}) |"
        f" {df_sub['pred'].nunique()} classes prédites"
    )

Fichier généré : Projet1_Script1_ML.csv ((1940, 2)) | 16 classes prédites
